# Chess DQN Agent - Training a Deep Q-Network to Play Chess

**Author:** Devin Williams  
**Course:** BYU CS474 - Deep Learning  
**Project:** Final Project - Reinforcement Learning for Chess

This notebook implements a Deep Q-Network (DQN) agent that learns to play chess through:
- Self-play reinforcement learning
- Supervised pretraining on human games from Lichess
- Experience replay and target networks for stable learning

---

**Quick Links:**
- Open in Colab: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/byu-cs474/blob/master/final_project/chess_dqn.ipynb)
- [Project Proposal](./project%20proposal.pdf)
- [Dataset Description](./Dataset%20Description.pdf)

## 1. Setup and Dependencies

Install required packages and verify GPU availability.

In [1]:
# Install dependencies
!pip install -q chess python-chess torch torchvision
!pip install -q zstandard  # For decompressing Lichess data (.zst files)
!pip install -q matplotlib seaborn tqdm

print("\n✓ Dependencies installed successfully!")

ERROR: Could not find a version that satisfies the requirement torch (from versions: none)
ERROR: No matching distribution found for torch

✓ Dependencies installed successfully!


In [2]:
# Import libraries
import chess
import chess.pgn
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import deque, namedtuple
import random
import io
import requests
import zstandard as zstd
from tqdm.auto import tqdm
import pickle
import os
from pathlib import Path
import json
from datetime import datetime

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print("✓ Imports successful!")

ModuleNotFoundError: No module named 'chess'

In [ ]:
# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ GPU not available. Training will be slower on CPU.")

In [ ]:
# Create directories for data and models
DATA_DIR = Path("chess_data")
MODEL_DIR = Path("models")
LOGS_DIR = Path("logs")

DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)

print("✓ Directories created!")

## 2. Lichess Data Download

Download chess games from the Lichess database. We'll start with a recent month's data.

Lichess provides free game databases at: https://database.lichess.org/

In [ ]:
def download_lichess_data(year=2016, month=2, max_size_mb=1500):
    """
    Download Lichess game database for a specific month.
    
    Args:
        year: Year of the database
        month: Month of the database (1-12)
        max_size_mb: Maximum size to download in MB (for safety)
    
    Returns:
        Path to the downloaded file
    """
    # Format: lichess_db_standard_rated_YYYY-MM.pgn.zst
    filename = f"lichess_db_standard_rated_{year}-{month:02d}.pgn.zst"
    url = f"https://database.lichess.org/standard/{filename}"
    filepath = DATA_DIR / filename
    
    # Check if already downloaded
    if filepath.exists():
        print(f"✓ File already exists: {filepath}")
        return filepath
    
    print(f"Downloading from: {url}")
    print(f"This may take several minutes...")
    
    try:
        # Stream download with progress bar
        response = requests.get(url, stream=True)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        total_size_mb = total_size / (1024 * 1024)
        
        print(f"File size: {total_size_mb:.2f} MB")
        
        if total_size_mb > max_size_mb:
            print(f"⚠ Warning: File is larger than {max_size_mb} MB.")
            print(f"Consider using a sample or different month.")
            response.close()
            return None
        
        # Download with progress bar
        with open(filepath, 'wb') as f:
            with tqdm(total=total_size, unit='B', unit_scale=True, desc=filename) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
        
        print(f"✓ Download complete: {filepath}")
        return filepath
        
    except requests.exceptions.RequestException as e:
        print(f"✗ Download failed: {e}")
        return None

In [ ]:
def decompress_and_sample_pgn(compressed_file, num_games=10000, output_file=None):
    """
    Decompress .zst file and extract a sample of games.
    
    Args:
        compressed_file: Path to .zst compressed PGN file
        num_games: Number of games to extract
        output_file: Optional path to save decompressed sample
    
    Returns:
        Path to decompressed file
    """
    if output_file is None:
        output_file = DATA_DIR / f"lichess_sample_{num_games}.pgn"
    
    # Check if sample already exists
    if output_file.exists():
        print(f"✓ Sample already exists: {output_file}")
        return output_file
    
    print(f"Decompressing and sampling {num_games} games...")
    
    dctx = zstd.ZstdDecompressor()
    games_extracted = 0
    
    with open(compressed_file, 'rb') as compressed:
        with dctx.stream_reader(compressed) as reader:
            text_stream = io.TextIOWrapper(reader, encoding='utf-8')
            with open(output_file, 'w', encoding='utf-8') as output:
                current_game = []
                
                for line in tqdm(text_stream, desc="Extracting games", total=num_games):
                    current_game.append(line)
                    
                    # Empty line indicates end of game
                    if line.strip() == "":
                        if current_game:
                            output.writelines(current_game)
                            games_extracted += 1
                            current_game = []
                            
                            if games_extracted >= num_games:
                                break
    
    print(f"✓ Extracted {games_extracted} games to {output_file}")
    return output_file

In [ ]:
# Download and prepare Lichess data
# Using February 2016 data (smaller, under 1 GB)

print("=" * 60)
print("LICHESS DATA DOWNLOAD")
print("=" * 60)

NUM_GAMES_TO_SAMPLE = 10000

print(f"\nDownloading Lichess database from February 2016...")
print(f"This older dataset is much smaller (under 1 GB).")
print(f"We'll extract a sample of {NUM_GAMES_TO_SAMPLE} games.\n")

# Download February 2016 data
compressed_file = download_lichess_data(year=2016, month=2, max_size_mb=1500)

if compressed_file:
    # Extract sample
    pgn_file = decompress_and_sample_pgn(compressed_file, num_games=NUM_GAMES_TO_SAMPLE)
    print(f"\n✓ Ready to parse PGN file: {pgn_file}")
else:
    print("\n⚠ Download failed or skipped. You can:")
    print("  1. Try a different month")
    print("  2. Manually download from https://database.lichess.org/")
    print("  3. Continue with self-play only (no pretraining data)")
    pgn_file = None

In [ ]:
def filter_game(game):
    """
    Filter games based on quality criteria.
    
    Args:
        game: chess.pgn.Game object
    
    Returns:
        True if game should be included, False otherwise
    """
    headers = game.headers
    
    # Check for required headers
    if 'Result' not in headers or 'TimeControl' not in headers:
        return False
    
    # Only include decided games (no aborted/unknown)
    result = headers['Result']
    if result not in ['1-0', '0-1', '1/2-1/2']:
        return False
    
    # Filter by time control (exclude bullet, include classical/rapid)
    time_control = headers['TimeControl']
    if time_control == '-':  # No time control
        return False
    
    # Parse time control (format: initial+increment)
    try:
        if '+' in time_control:
            initial_time = int(time_control.split('+')[0])
        else:
            initial_time = int(time_control)
        
        # Exclude bullet (< 180 seconds)
        if initial_time < 180:
            return False
    except:
        return False
    
    # Check for minimum number of moves (avoid very short games)
    num_moves = len(list(game.mainline_moves()))
    if num_moves < 10:
        return False
    
    return True

In [ ]:
def parse_game_result(result_string):
    """
    Convert PGN result string to reward values.
    
    Args:
        result_string: '1-0', '0-1', or '1/2-1/2'
    
    Returns:
        Tuple of (white_reward, black_reward)
    """
    if result_string == '1-0':
        return (1.0, -1.0)  # White wins
    elif result_string == '0-1':
        return (-1.0, 1.0)  # Black wins
    elif result_string == '1/2-1/2':
        return (0.0, 0.0)   # Draw
    else:
        return (0.0, 0.0)   # Unknown/aborted

In [ ]:
def parse_pgn_file(pgn_file, max_games=None):
    """
    Parse PGN file and extract games.
    
    Args:
        pgn_file: Path to PGN file
        max_games: Maximum number of games to parse (None for all)
    
    Returns:
        List of parsed games with metadata
    """
    games_data = []
    
    with open(pgn_file, 'r', encoding='utf-8') as f:
        game_count = 0
        filtered_count = 0
        
        pbar = tqdm(desc="Parsing games", total=max_games)
        
        while True:
            game = chess.pgn.read_game(f)
            
            if game is None:
                break
            
            game_count += 1
            
            # Apply filters
            if not filter_game(game):
                filtered_count += 1
                continue
            
            # Extract game data
            result = game.headers.get('Result', '*')
            white_reward, black_reward = parse_game_result(result)
            
            # Store moves and board states
            board = game.board()
            moves = []
            
            for move in game.mainline_moves():
                # Store state before move
                fen = board.fen()
                moves.append({
                    'fen': fen,
                    'move': move.uci(),
                    'turn': board.turn  # True = White, False = Black
                })
                board.push(move)
            
            games_data.append({
                'moves': moves,
                'result': result,
                'white_reward': white_reward,
                'black_reward': black_reward,
                'num_moves': len(moves)
            })
            
            pbar.update(1)
            
            if max_games and len(games_data) >= max_games:
                break
        
        pbar.close()
    
    print(f"\nParsing complete:")
    print(f"  Total games read: {game_count}")
    print(f"  Games filtered out: {filtered_count}")
    print(f"  Games included: {len(games_data)}")
    
    return games_data

In [ ]:
# Parse the downloaded PGN file
print("=" * 60)
print("PARSING PGN DATA")
print("=" * 60)

# Check if we have a PGN file to parse
pgn_files = list(DATA_DIR.glob("*.pgn"))

if pgn_files:
    # Use the most recent sample file
    pgn_file = pgn_files[0]
    print(f"\nParsing: {pgn_file}\n")
    
    # Parse games (limit to prevent memory issues in Colab)
    parsed_games = parse_pgn_file(pgn_file, max_games=5000)
    
    # Save parsed data for later use
    parsed_data_file = DATA_DIR / "parsed_games.pkl"
    with open(parsed_data_file, 'wb') as f:
        pickle.dump(parsed_games, f)
    
    print(f"\n✓ Parsed data saved to: {parsed_data_file}")
    
    # Display sample game statistics
    if parsed_games:
        print("\nSample Game Statistics:")
        avg_moves = np.mean([g['num_moves'] for g in parsed_games])
        print(f"  Average moves per game: {avg_moves:.1f}")
        
        results = [g['result'] for g in parsed_games]
        print(f"  White wins: {results.count('1-0')}")
        print(f"  Black wins: {results.count('0-1')}")
        print(f"  Draws: {results.count('1/2-1/2')}")
else:
    print("\n⚠ No PGN files found in data directory.")
    print("Continuing without pretraining data (will use self-play only).")
    parsed_games = []

## 4. Board State Encoding

Convert chess board positions to 8×8×12 tensor representation.

In [ ]:
# Placeholder for next section
print("Board state encoding will be implemented next...")

---
## Next Steps

Upcoming sections:
- Board state encoding (8×8×12 tensor)
- Chess environment wrapper
- DQN architecture
- Training loop
- Evaluation and play interface